# Notebook de Preparação dos Dados para a Pipeline do Modelo

Este notebook reúne os passos necessários para explorar, organizar e preparar os dados de coordenadas das mãos capturados com o **HandLandmark do MediaPipe**.

Nosso objetivo é transformar essas coordenadas em um conjunto de dados consistente e bem estruturado, pronto para treinar modelos de **machine learning** capazes de classificar letras da **Libras** com maior precisão.

Ao longo do notebook, vamos seguir uma sequência clara de trabalho: entender a origem dos dados, inspecionar sua estrutura, tratar possíveis inconsistências e preparar tudo para a etapa de treinamento do modelo.

## Passo a passo 

1. **Carregar os dados**
   - Importar os arquivos ou tabelas com as coordenadas das mãos.
   - Verificar se os dados foram lidos corretamente.

2. **Entender a estrutura do conjunto de dados**
   - Conferir colunas, tipos de dados e quantidade de registros.
   - Identificar quais atributos representam os pontos da mão.

3. **Fazer uma análise inicial**
   - Verificar valores ausentes, dados duplicados e possíveis inconsistências.
   - Observar a distribuição das classes, como letras ou sinais.

4. **Limpar e organizar os dados**
   - Tratar valores nulos e remover duplicatas, se necessário.
   - Padronizar nomes de colunas e formatos.

5. **Engenharia de Features**
   - Calcular a distância euclidiana entre as pontas dos dedos e o pulso.

6. **Preparar as variáveis para o modelo**
   - Separar entrada e saída.
   - Manter as coordenadas e as novas distâncias como variáveis de entrada.

7. **Dividir os dados para treino e teste**
   - Criar conjuntos de treino, validação e teste.
   - Garantir que a divisão seja representativa.

8. **Salvar a versão final preparada**
   - Exportar o dataset tratado para uso na pipeline de treinamento.
   - Registrar decisões importantes tomadas durante a preparação.

### Imports 

In [ ]:
# Imports
import json
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep")

# Caminhos dos arquivos
# Funciona tanto se o notebook for executado a partir da raiz do projeto
# quanto se for executado de dentro da pasta notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_separated = PROJECT_ROOT / "data" / "data_separeted"
data_output = PROJECT_ROOT / "data" / "data_output"

RANDOM_STATE = 42
LABEL_COLUMN = "label"


## 1. **Carregar os dados**
   - Importar os arquivos ou tabelas com as coordenadas das mãos.
   - Verificar se os dados foram lidos corretamente.


In [ ]:
def concat_tables(data_input, data_output):
    dfs = []

    for item in sorted(data_input.glob("*.csv")):
        df_temp = pd.read_csv(item)
        df_temp["source_file"] = item.name
        dfs.append(df_temp)

    if not dfs:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em {data_input}")

    df = pd.concat(dfs, ignore_index=True)

    data_output.mkdir(parents=True, exist_ok=True)
    output_file = data_output / "dataset_concatenado.csv"
    df.to_csv(output_file, index=False)

    return df


In [ ]:
df = concat_tables(data_separated, data_output)
df.head()


## 2. **Entender a estrutura do conjunto de dados**
   - Conferir colunas, tipos de dados e quantidade de registros.
   - Identificar quais atributos representam os pontos da mão.


In [ ]:
landmark_columns = [col for col in df.columns if col != LABEL_COLUMN and col != "source_file"]
expected_landmark_columns = [f"{axis}{idx}" for idx in range(21) for axis in ("x", "y", "z")]
missing_landmark_columns = sorted(set(expected_landmark_columns) - set(landmark_columns))
extra_landmark_columns = sorted(set(landmark_columns) - set(expected_landmark_columns))

estrutura = pd.DataFrame(
    {
        "metrica": [
            "linhas",
            "colunas",
            "classes",
            "features_landmarks",
            "colunas_landmarks_ausentes",
            "colunas_landmarks_extras",
        ],
        "valor": [
            len(df),
            df.shape[1],
            df[LABEL_COLUMN].nunique(),
            len(landmark_columns),
            missing_landmark_columns,
            extra_landmark_columns,
        ],
    }
)

estrutura


In [ ]:
df.dtypes.value_counts()


## 3. **Fazer uma análise inicial**
   - Verificar valores ausentes, dados duplicados e possíveis inconsistências.
   - Observar a distribuição das classes, como letras ou sinais.


In [ ]:
qualidade = pd.DataFrame(
    {
        "metrica": [
            "valores_ausentes",
            "linhas_duplicadas",
            "labels_ausentes",
            "arquivos_origem",
        ],
        "valor": [
            int(df.isna().sum().sum()),
            int(df.duplicated().sum()),
            int(df[LABEL_COLUMN].isna().sum()),
            df["source_file"].nunique(),
        ],
    }
)

qualidade


In [ ]:
class_counts = df[LABEL_COLUMN].value_counts().sort_index()
class_counts.to_frame("quantidade")


In [ ]:
plt.figure(figsize=(12, 4))
sns.countplot(data=df, x=LABEL_COLUMN, order=class_counts.index)
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. **Limpar e organizar os dados**
   - Tratar valores nulos e remover duplicatas, se necessário.
   - Padronizar nomes de colunas e formatos.


In [ ]:
def clean_dataset(df, label_column=LABEL_COLUMN):
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.strip()

    df_clean[label_column] = (
        df_clean[label_column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    feature_columns = [col for col in df_clean.columns if col not in [label_column, "source_file"]]
    df_clean[feature_columns] = df_clean[feature_columns].apply(pd.to_numeric, errors="coerce")

    rows_before = len(df_clean)
    df_clean = df_clean.dropna(subset=[label_column, *feature_columns])
    df_clean = df_clean.reset_index(drop=True)

    cleaning_report = {
        "linhas_antes": rows_before,
        "linhas_depois": len(df_clean),
        "linhas_removidas": rows_before - len(df_clean),
        "valores_ausentes_restantes": int(df_clean[[label_column, *feature_columns]].isna().sum().sum()),
    }

    return df_clean, feature_columns, cleaning_report


df_clean, feature_columns, cleaning_report = clean_dataset(df)
cleaning_report


In [ ]:
df_clean[LABEL_COLUMN].value_counts().sort_index().to_frame("quantidade")


## 5. **Engenharia de Features**
   - Calcular a distância euclidiana entre as pontas dos dedos e o pulso.

In [ ]:
fingertips = {'polegar': 4, 'indicador': 8, 'medio': 12, 'anelar': 16, 'minimo': 20}
for name, idx in fingertips.items():
    col_name = f'dist_{name}_pulso'
    df_clean[col_name] = ((df_clean[f'x{idx}'] - df_clean['x0'])**2 + \
                         (df_clean[f'y{idx}'] - df_clean['y0'])**2 + \
                         (df_clean[f'z{idx}'] - df_clean['z0'])**2)**0.5
    feature_columns.append(col_name)

## 6. **Preparar as variáveis para o modelo**
   - Separar entrada e saída.
   - Manter as coordenadas e as novas distâncias como variáveis de entrada.


In [ ]:
X = df_clean[feature_columns]
y = df_clean[LABEL_COLUMN]

print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(f"Classes: {sorted(y.unique())}")


## 7. **Dividir os dados para treino, validação e teste**
   - Criar conjuntos estratificados para preservar a proporção das classes.
   - Calcular estatísticas de padronização usando apenas o treino.


In [ ]:
def stratified_split(df, label_column=LABEL_COLUMN, test_size=0.15, val_size=0.15, random_state=RANDOM_STATE):
    def sample_by_class(input_df, frac, seed):
        sampled_groups = []

        for _, group in input_df.groupby(label_column):
            sampled_groups.append(group.sample(frac=frac, random_state=seed))

        return pd.concat(sampled_groups).sort_index()

    test = sample_by_class(df, test_size, random_state)
    remaining = df.drop(test.index)

    val_fraction_from_remaining = val_size / (1 - test_size)
    val = sample_by_class(remaining, val_fraction_from_remaining, random_state)
    train = remaining.drop(val.index)

    return (
        train.sample(frac=1, random_state=random_state).reset_index(drop=True),
        val.sample(frac=1, random_state=random_state).reset_index(drop=True),
        test.sample(frac=1, random_state=random_state).reset_index(drop=True),
    )


train_df, val_df, test_df = stratified_split(df_clean)
train_df = train_df.drop_duplicates(subset=[LABEL_COLUMN, *feature_columns]).reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "split": ["treino", "validacao", "teste"],
        "linhas": [len(train_df), len(val_df), len(test_df)],
        "percentual": [len(train_df) / len(df_clean), len(val_df) / len(df_clean), len(test_df) / len(df_clean)],
    }
)

split_summary

In [ ]:
pd.concat(
    {
        "treino": train_df[LABEL_COLUMN].value_counts(normalize=True).sort_index(),
        "validacao": val_df[LABEL_COLUMN].value_counts(normalize=True).sort_index(),
        "teste": test_df[LABEL_COLUMN].value_counts(normalize=True).sort_index(),
    },
    axis=1,
).fillna(0).round(4)


In [ ]:
def standardize_splits(train_df, val_df, test_df, feature_columns):
    train_mean = train_df[feature_columns].mean()
    train_std = train_df[feature_columns].std().replace(0, 1)

    def transform(split_df):
        split_scaled = split_df.copy()
        split_scaled[feature_columns] = (split_scaled[feature_columns] - train_mean) / train_std
        return split_scaled

    scaler_stats = pd.DataFrame(
        {
            "feature": feature_columns,
            "mean": train_mean.values,
            "std": train_std.values,
        }
    )

    return transform(train_df), transform(val_df), transform(test_df), scaler_stats


train_scaled, val_scaled, test_scaled, scaler_stats = standardize_splits(
    train_df,
    val_df,
    test_df,
    feature_columns,
)

train_scaled.head()


## 8. **Salvar a versão final preparada**
   - Exportar o dataset tratado para uso na pipeline de treinamento.
   - Registrar decisões importantes tomadas durante a preparação.


In [ ]:
prepared_output = data_output / "prepared"
prepared_output.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(prepared_output / "dataset_limpo.csv", index=False)
train_scaled.to_csv(prepared_output / "train.csv", index=False)
val_scaled.to_csv(prepared_output / "validation.csv", index=False)
test_scaled.to_csv(prepared_output / "test.csv", index=False)
scaler_stats.to_csv(prepared_output / "standardization_stats.csv", index=False)

metadata = {
    "label_column": LABEL_COLUMN,
    "feature_columns": feature_columns,
    "classes": sorted(df_clean[LABEL_COLUMN].unique().tolist()),
    "random_state": RANDOM_STATE,
    "split_rows": {
        "train": len(train_scaled),
        "validation": len(val_scaled),
        "test": len(test_scaled),
    },
    "cleaning_report": cleaning_report,
    "outputs": {
        "clean_dataset": str(prepared_output / "dataset_limpo.csv"),
        "train": str(prepared_output / "train.csv"),
        "validation": str(prepared_output / "validation.csv"),
        "test": str(prepared_output / "test.csv"),
        "standardization_stats": str(prepared_output / "standardization_stats.csv"),
    },
}

with (prepared_output / "metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

metadata
